In [ ]:
import os
os.chdir(os.path.join('..', 'results'))

# Statistical Significance Analysis
**Input:** 6 pkl files from 30-run experiments
**Tests:** Friedman + Wilcoxon post-hoc (Bonferroni correction)
**Note:**
- S1A pkl structure: `clustering_results_30[ds][method]`  
- S1B pkl structure: `clustering_results_30[ds][method]`  
- S2 pkl structure: `final_results_30[ds][method]`  
- S2A/S2B each split into kmf + ccf → must merge methods per dataset


## Cell 1: Imports

In [3]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import scikit_posthocs as sp
import warnings; warnings.filterwarnings('ignore')
print("Imports OK")

Imports OK


## Cell 2: Load pkl Files

In [5]:
# FIXED: file names now match what the experiment notebooks save
# (previously S2A_kmf / S2B_ccf were read without '-1', i.e. stale older files)
with open('Strategy1_30runs_PartA.pkl', 'rb') as f: s1a = pickle.load(f)
with open('Strategy1_30runs_PartB.pkl', 'rb') as f: s1b = pickle.load(f)
with open('S2A_results_kmf_30runs-1.pkl', 'rb') as f: s2a_kmf = pickle.load(f)
with open('S2A_results_ccf_30runs-1.pkl', 'rb') as f: s2a_ccf = pickle.load(f)
with open('S2B_results_kmf_30runs-1.pkl', 'rb') as f: s2b_kmf = pickle.load(f)
with open('S2B_results_ccf_30runs-1.pkl', 'rb') as f: s2b_ccf = pickle.load(f)
print("All loaded ✓")

All loaded ✓


## Cell 3: Extract & Merge Data

In [7]:
# ORIGINAL (Reviewer 3, Comment 5 & 6 -- kept for reference, not executed):
# UTIL_COLS = ['Inertia','Silhouette','Calinski_Harabasz','Davies_Bouldin','BCSS','Variance_Ratio']
# FAIR_COLS = ['Balance','SPD','Disparate_Impact','Entropy']
# ALL_COLS  = UTIL_COLS + FAIR_COLS
# HIGHER    = {'Silhouette','Calinski_Harabasz','BCSS','Variance_Ratio','Balance','Entropy'}
#
# N_PAIRS = len(list(combinations(METHODS, 2)))   # 21
# ALPHA   = 0.05
# ALPHA_B = ALPHA / N_PAIRS                        # ≈ 0.00238
#
# def extract_s1(pkl):
#     results = pkl['clustering_results_30']
#     first_key = list(results.keys())[0]
#     if first_key in ['approach1', 'approach2']:
#         results = results.get('approach2', results.get('approach1', {}))
#     out = {}
#     for ds, methods in results.items():
#         out[ds] = {}
#         for m, r in methods.items():
#             out[ds][m] = {**r['mean_quality'], **r['mean_fairness']}
#     return out
#
# def extract_s2(pkl):
#     results = pkl['final_results_30']
#     out = {}
#     for ds, methods in results.items():
#         out[ds] = {}
#         for m, r in methods.items():
#             out[ds][m.lower()] = {**r['mean_quality'], **r['mean_fairness']}
#     return out

METHOD_LABEL = {
    'kmeans':              'K-Means',
    'fairlet':             'Fairlet',
    'bfkm':               'BFKM',
    'fair_centroid':       'CCF',
    'postprocessing_nfp':  'PP-NFP',
    'postprocessing_gini': 'PP-Gini',
    'rawlsian':            'Rawlsian',
}
METHODS   = list(METHOD_LABEL.keys())
M_LABELS  = list(METHOD_LABEL.values())

# FIXED (Reviewer 3, Comment 6): Calinski_Harabasz and BCSS removed --
# algebraically redundant with Variance_Ratio and Inertia respectively
# under fixed k (see Strategy1_PartA_30runs.ipynb for the derivation).
UTIL_COLS = ['Inertia','Silhouette','Davies_Bouldin','Variance_Ratio']
# FIXED (Reviewer 3, Comment 2): 'DI_dist' = |DI-1| used for ranking/testing
# instead of raw 'Disparate_Impact' (ideal = 1, not 0).
FAIR_COLS = ['Balance','SPD','DI_dist','Entropy']
ALL_COLS  = UTIL_COLS + FAIR_COLS
HIGHER    = {'Silhouette','Variance_Ratio','Balance','Entropy'}

N_PAIRS = len(list(combinations(METHODS, 2)))   # 21
ALPHA   = 0.05
ALPHA_B = ALPHA / N_PAIRS                        # ≈ 0.00238
# ADDED (Reviewer 3, Comment 5): "No correction is applied across the
# multiple omnibus tests." Bonferroni correction across the len(ALL_COLS)
# Friedman omnibus tests run below (one per metric).
ALPHA_OMNIBUS = ALPHA / len(ALL_COLS)

# ADDED (Reviewer 3, Comment 5): maps a dataset configuration to its base
# dataset (e.g. 'adult_race' and 'adult_combined' both -> 'adult'), needed
# for the hierarchical analysis, since "the 18 configurations are not
# independent because five base datasets are repeated under different
# sensitive-attribute definitions."
def base_dataset(ds_name):
    for suffix in ('_combined', '_race', '_gender'):
        if ds_name.endswith(suffix):
            return ds_name[:-len(suffix)]
    return ds_name

# ── Extract from S1 pkl ──────────────────────────────────────────────────────
def extract_s1(pkl):
    """
    S1A/S1B structure:
      pkl['clustering_results_30'][ds][method]
        -> r['mean_quality'], r['mean_fairness'], r['all_quality'], r['all_fairness']
    Old S1A may have ['clustering_results_30']['approach2'][ds][method]

    FIXED (Reviewer 3, Comment 5): now also returns the raw per-run values
    ('all_quality'/'all_fairness', one dict per seed) alongside the means,
    needed for the run-level analysis below -- "the Friedman and Wilcoxon
    tests use one averaged value per configuration, discarding run-level
    uncertainty."
    """
    results = pkl['clustering_results_30']
    first_key = list(results.keys())[0]
    if first_key in ['approach1', 'approach2']:
        results = results.get('approach2', results.get('approach1', {}))
    out, out_runs = {}, {}
    for ds, methods in results.items():
        out[ds] = {}
        out_runs[ds] = {}
        for m, r in methods.items():
            di_raw = r['mean_fairness'].get('Disparate_Impact', np.nan)
            mean_fairness = {**r['mean_fairness'], 'DI_dist': abs(di_raw - 1.0)}
            out[ds][m] = {**r['mean_quality'], **mean_fairness}
            runs_q = r.get('all_quality', [])
            runs_f = r.get('all_fairness', [])
            n_runs = min(len(runs_q), len(runs_f))
            per_run = []
            for i in range(n_runs):
                di_raw_i = runs_f[i].get('Disparate_Impact', np.nan)
                per_run.append({**runs_q[i], **runs_f[i], 'DI_dist': abs(di_raw_i - 1.0)})
            out_runs[ds][m] = per_run
    return out, out_runs

# ── Extract from S2 pkl ──────────────────────────────────────────────────────
def extract_s2(pkl):
    """
    S2 structure: pkl['final_results_30'][ds][method]
      -> r['mean_quality'], r['mean_fairness'], r['all_quality'], r['all_fairness']

    FIXED (Reviewer 3, Comment 5): same run-level extraction as extract_s1.
    """
    results = pkl['final_results_30']
    out, out_runs = {}, {}
    for ds, methods in results.items():
        out[ds] = {}
        out_runs[ds] = {}
        for m, r in methods.items():
            m_lower = m.lower()
            di_raw = r['mean_fairness'].get('Disparate_Impact', np.nan)
            mean_fairness = {**r['mean_fairness'], 'DI_dist': abs(di_raw - 1.0)}
            out[ds][m_lower] = {**r['mean_quality'], **mean_fairness}
            runs_q = r.get('all_quality', [])
            runs_f = r.get('all_fairness', [])
            n_runs = min(len(runs_q), len(runs_f))
            per_run = []
            for i in range(n_runs):
                di_raw_i = runs_f[i].get('Disparate_Impact', np.nan)
                per_run.append({**runs_q[i], **runs_f[i], 'DI_dist': abs(di_raw_i - 1.0)})
            out_runs[ds][m_lower] = per_run
    return out, out_runs

# ORIGINAL (Reviewer 3, Comment 5 -- kept for reference, not executed):
# s1_data = {}
# for pkl in [s1a, s1b]:
#     for ds, methods in extract_s1(pkl).items():
#         if ds not in s1_data:
#             s1_data[ds] = {}
#         s1_data[ds].update(methods)
#
# s2_data = {}
# for pkl in [s2a_kmf, s2a_ccf, s2b_kmf, s2b_ccf]:
#     for ds, methods in extract_s2(pkl).items():
#         if ds not in s2_data:
#             s2_data[ds] = {}
#         s2_data[ds].update(methods)

# FIXED (Reviewer 3, Comment 5): extract_s1/extract_s2 now return
# (means_dict, runs_dict) tuples; both are merged here so the run-level
# raw values (s1_runs/s2_runs) are available alongside the existing
# mean-based dicts (s1_data/s2_data).
s1_data, s1_runs = {}, {}
for pkl in [s1a, s1b]:
    means, runs = extract_s1(pkl)
    for ds, methods in means.items():
        s1_data.setdefault(ds, {}).update(methods)
    for ds, methods in runs.items():
        s1_runs.setdefault(ds, {}).update(methods)

# ── Merge S2 — must merge methods per dataset, not overwrite ─────────────────
s2_data, s2_runs = {}, {}
for pkl in [s2a_kmf, s2a_ccf, s2b_kmf, s2b_ccf]:
    means, runs = extract_s2(pkl)
    for ds, methods in means.items():
        s2_data.setdefault(ds, {}).update(methods)   # merge methods, not overwrite dataset
    for ds, methods in runs.items():
        s2_runs.setdefault(ds, {}).update(methods)

print(f"S1 datasets: {len(s1_data)}  methods per ds: {len(list(s1_data.values())[0])}")
print(f"S2 datasets: {len(s2_data)}  methods per ds: {len(list(s2_data.values())[0])}")
print(f"N_pairs: {N_PAIRS}, Bonferroni alpha (pairwise): {ALPHA_B:.5f}, Bonferroni alpha (omnibus): {ALPHA_OMNIBUS:.5f}")

# Sanity check
first_s2 = list(s2_data.keys())[0]
print(f"\nS2 first dataset '{first_s2}' methods: {list(s2_data[first_s2].keys())}")
print(f"S1 run-level data available: {all(len(v)>0 for ds in s1_runs for v in s1_runs[ds].values())}")
print(f"S2 run-level data available: {all(len(v)>0 for ds in s2_runs for v in s2_runs[ds].values())}")

S1 datasets: 18  methods per ds: 7
S2 datasets: 18  methods per ds: 7
N_pairs: 21, Bonferroni alpha (pairwise): 0.00238, Bonferroni alpha (omnibus): 0.00625

S2 first dataset 'adult' methods: ['kmeans', 'fairlet', 'bfkm', 'fair_centroid', 'postprocessing_nfp', 'postprocessing_gini', 'rawlsian']
S1 run-level data available: True
S2 run-level data available: True


## Cell 4: Build Score Matrix

In [9]:
def build_score_matrix(data, metric):
    """
    DataFrame: rows=datasets, cols=methods.
    Only include datasets where ALL 7 methods have a value for this metric.
    """
    complete_ds = []
    for ds in data:
        has_all = all(
            m in data[ds] and metric in data[ds][m]
            and not np.isnan(data[ds][m][metric])
            for m in METHODS
        )
        if has_all:
            complete_ds.append(ds)

    if not complete_ds:
        return pd.DataFrame()

    rows = []
    for ds in complete_ds:
        row = {'Dataset': ds}
        for m in METHODS:
            row[m] = data[ds][m][metric]
        rows.append(row)
    return pd.DataFrame(rows).set_index('Dataset')

# Check coverage
print("=== Dataset coverage per metric (S1) ===")
for col in ALL_COLS:
    mat = build_score_matrix(s1_data, col)
    print(f"  {col:25s}: {len(mat)} complete datasets")

print("\n=== Dataset coverage per metric (S2) ===")
for col in ALL_COLS:
    mat = build_score_matrix(s2_data, col)
    print(f"  {col:25s}: {len(mat)} complete datasets")

=== Dataset coverage per metric (S1) ===
  Inertia                  : 18 complete datasets
  Silhouette               : 18 complete datasets
  Davies_Bouldin           : 18 complete datasets
  Variance_Ratio           : 18 complete datasets
  Balance                  : 18 complete datasets
  SPD                      : 18 complete datasets
  DI_dist                  : 18 complete datasets
  Entropy                  : 18 complete datasets

=== Dataset coverage per metric (S2) ===
  Inertia                  : 18 complete datasets
  Silhouette               : 18 complete datasets
  Davies_Bouldin           : 18 complete datasets
  Variance_Ratio           : 18 complete datasets
  Balance                  : 18 complete datasets
  SPD                      : 18 complete datasets
  DI_dist                  : 18 complete datasets
  Entropy                  : 18 complete datasets


## Cell 5: Friedman Test

In [11]:
# ORIGINAL (Reviewer 3, Comment 5 -- kept for reference, not executed):
# def run_friedman(data, metrics):
#     rows = []
#     for metric in metrics:
#         mat = build_score_matrix(data, metric)
#         if mat.empty or len(mat) < 3:
#             rows.append({'Metric': metric, 'N_datasets': len(mat) if not mat.empty else 0,
#                          'Friedman_stat': 'N/A', 'p_value': 'N/A', 'Significant': 'N/A'})
#             continue
#         arrays = [mat[m].values for m in METHODS]
#         try:
#             stat, p = friedmanchisquare(*arrays)
#             sig = 'YES *' if p < ALPHA else 'no'
#         except Exception:
#             stat, p, sig = np.nan, np.nan, 'error'
#         rows.append({
#             'Metric':        metric,
#             'N_datasets':    len(mat),
#             'Friedman_stat': round(stat, 3) if not np.isnan(stat) else 'N/A',
#             'p_value':       round(p, 6)    if not np.isnan(p)    else 'N/A',
#             'Significant':   sig
#         })
#     return pd.DataFrame(rows)
#
# friedman_s1 = run_friedman(s1_data, ALL_COLS)
# friedman_s2 = run_friedman(s2_data, ALL_COLS)

# ADDED (Reviewer 3, Comment 5): Kendall's W effect size for the Friedman
# test, computed from the chi-square statistic: W = chi2 / (n*(k-1)),
# where n = number of blocks (datasets) and k = number of treatments
# (methods). W ranges 0 (no agreement/effect) to 1 (perfect agreement).
def kendalls_w(chi2_stat, n_blocks, k_treatments):
    if n_blocks is None or k_treatments is None or n_blocks < 1 or k_treatments < 2:
        return np.nan
    return chi2_stat / (n_blocks * (k_treatments - 1))

# ADDED (Reviewer 3, Comment 5): hierarchical aggregation -- "the 18
# configurations are not independent because five base datasets are
# repeated under different sensitive-attribute definitions...An analysis
# that first aggregates within each base dataset would be more
# appropriate." Averages each metric across all sensitive-attribute
# variants of the same base dataset before testing, so the Friedman test's
# "blocks" are genuinely independent base datasets, not correlated
# re-uses of the same underlying data.
def build_hierarchical_matrix(data, metric):
    mat = build_score_matrix(data, metric)
    if mat.empty:
        return mat
    mat = mat.copy()
    mat['_base'] = [base_dataset(ds) for ds in mat.index]
    return mat.groupby('_base')[METHODS].mean()

# ADDED (Reviewer 3, Comment 5): run-level analysis -- "the Friedman and
# Wilcoxon tests use one averaged value per configuration, discarding
# run-level uncertainty." Runs an independent Friedman test on each of the
# 30 seeds' raw per-run values (one score matrix per seed, not a single
# matrix of means), then combines the 30 resulting p-values into one
# overall p-value via Stouffer's method (scipy.stats.combine_pvalues).
def build_score_matrix_for_run(runs_data, metric, run_idx):
    complete_ds = []
    for ds in runs_data:
        has_all = all(
            m in runs_data[ds] and run_idx < len(runs_data[ds][m])
            and metric in runs_data[ds][m][run_idx]
            and not np.isnan(runs_data[ds][m][run_idx][metric])
            for m in METHODS
        )
        if has_all:
            complete_ds.append(ds)
    if not complete_ds:
        return pd.DataFrame()
    rows = []
    for ds in complete_ds:
        row = {'Dataset': ds}
        for m in METHODS:
            row[m] = runs_data[ds][m][run_idx][metric]
        rows.append(row)
    return pd.DataFrame(rows).set_index('Dataset')

def run_level_friedman(runs_data, metric, n_runs=30):
    from scipy.stats import combine_pvalues
    pvals, stats_list = [], []
    for run_idx in range(n_runs):
        mat = build_score_matrix_for_run(runs_data, metric, run_idx)
        if mat.empty or len(mat) < 3:
            continue
        arrays = [mat[m].values for m in METHODS]
        try:
            stat, p = friedmanchisquare(*arrays)
            pvals.append(p); stats_list.append(stat)
        except Exception:
            continue
    if len(pvals) < 2:
        return np.nan, np.nan, len(pvals)
    combined_stat, combined_p = combine_pvalues(pvals, method='stouffer')
    return combined_p, np.mean(stats_list), len(pvals)

# FIXED (Reviewer 3, Comment 5): run_friedman now reports three parallel
# analyses per metric -- (a) the original mean-based test across 18
# configurations, (b) a hierarchical test that first aggregates within
# each base dataset, and (c) a run-level test combining 30 per-seed
# Friedman tests via Stouffer's method -- alongside Kendall's W effect
# size and a Bonferroni-corrected omnibus significance threshold
# (ALPHA_OMNIBUS) applied across all len(ALL_COLS) metrics tested.
def run_friedman(data, runs_data, metrics):
    rows = []
    for metric in metrics:
        mat = build_score_matrix(data, metric)
        if mat.empty or len(mat) < 3:
            rows.append({'Metric': metric, 'N_datasets': len(mat) if not mat.empty else 0,
                         'Friedman_stat': 'N/A', 'p_value': 'N/A', 'Kendalls_W': 'N/A',
                         'Significant_Bonf': 'N/A', 'Hier_p': 'N/A', 'Hier_N': 'N/A',
                         'RunLevel_p': 'N/A', 'RunLevel_N': 'N/A'})
            continue
        arrays = [mat[m].values for m in METHODS]
        try:
            stat, p = friedmanchisquare(*arrays)
            w = kendalls_w(stat, len(mat), len(METHODS))
            sig = 'YES *' if p < ALPHA_OMNIBUS else 'no'
        except Exception:
            stat, p, w, sig = np.nan, np.nan, np.nan, 'error'

        hier_mat = build_hierarchical_matrix(data, metric)
        if not hier_mat.empty and len(hier_mat) >= 3:
            try:
                hier_stat, hier_p = friedmanchisquare(*[hier_mat[m].values for m in METHODS])
            except Exception:
                hier_p = np.nan
        else:
            hier_p = np.nan

        run_p, run_stat_mean, run_n = run_level_friedman(runs_data, metric)

        rows.append({
            'Metric':          metric,
            'N_datasets':      len(mat),
            'Friedman_stat':   round(stat, 3) if not np.isnan(stat) else 'N/A',
            'p_value':         round(p, 6)    if not np.isnan(p)    else 'N/A',
            'Kendalls_W':      round(w, 3)    if not np.isnan(w)    else 'N/A',
            'Significant_Bonf':sig,
            'Hier_N':          len(hier_mat) if not hier_mat.empty else 0,
            'Hier_p':          round(hier_p, 6) if not np.isnan(hier_p) else 'N/A',
            'RunLevel_N':      run_n,
            'RunLevel_p':      round(run_p, 6) if not np.isnan(run_p) else 'N/A',
        })
    return pd.DataFrame(rows)

friedman_s1 = run_friedman(s1_data, s1_runs, ALL_COLS)
friedman_s2 = run_friedman(s2_data, s2_runs, ALL_COLS)

print("=" * 65)
print(f"FRIEDMAN TEST -- Strategy 1  (Bonferroni omnibus alpha = {ALPHA_OMNIBUS:.5f})")
print("=" * 65)
print(friedman_s1.to_string(index=False))

print("\n" + "=" * 65)
print(f"FRIEDMAN TEST -- Strategy 2  (Bonferroni omnibus alpha = {ALPHA_OMNIBUS:.5f})")
print("=" * 65)
print(friedman_s2.to_string(index=False))

FRIEDMAN TEST -- Strategy 1  (Bonferroni omnibus alpha = 0.00625)
        Metric  N_datasets  Friedman_stat  p_value  Kendalls_W Significant_Bonf  Hier_N   Hier_p  RunLevel_N  RunLevel_p
       Inertia          18         72.415 0.000000       0.671            YES *       8 0.000026          30         0.0
    Silhouette          18         37.463 0.000001       0.347            YES *       8 0.004118          30         0.0
Davies_Bouldin          18         64.268 0.000000       0.595            YES *       8 0.000047          30         0.0
Variance_Ratio          18         71.561 0.000000       0.663            YES *       8 0.000028          30         0.0
       Balance          18         24.987 0.000343       0.231            YES *       8 0.114026          30         0.0
           SPD          18         26.982 0.000146       0.250            YES *       8 0.005469          30         0.0
       DI_dist          18         19.953 0.002823       0.185            YES *       8

## Cell 6: Wilcoxon Post-hoc + Bonferroni

In [13]:
# ORIGINAL (Reviewer 3, Comment 5 -- kept for reference, not executed):
# def run_wilcoxon(data, metric):
#     mat = build_score_matrix(data, metric)
#     if mat.empty:
#         return pd.DataFrame()
#     pairs = list(combinations(METHODS, 2))
#     rows  = []
#     for m1, m2 in pairs:
#         x = mat[m1].values
#         y = mat[m2].values
#         n = min(len(x), len(y))
#         if n < 3:
#             rows.append({'Method_A': METHOD_LABEL[m1], 'Method_B': METHOD_LABEL[m2],
#                          'W_stat': '-', 'p_raw': '-', 'p_bonf': '-',
#                          f'Sig(p<{ALPHA_B:.4f})': '-'})
#             continue
#         try:
#             stat, p   = wilcoxon(x[:n], y[:n], zero_method='wilcox')
#             p_bonf    = min(p * N_PAIRS, 1.0)
#             sig       = 'YES *' if p < ALPHA_B else 'no'
#         except Exception:
#             stat = p = p_bonf = np.nan; sig = '-'
#         rows.append({
#             'Method_A':              METHOD_LABEL[m1],
#             'Method_B':              METHOD_LABEL[m2],
#             'W_stat':                round(stat, 3)   if not np.isnan(stat)   else '-',
#             'p_raw':                 round(p, 6)      if not np.isnan(p)      else '-',
#             'p_bonf':                round(p_bonf, 6) if not np.isnan(p_bonf) else '-',
#             f'Sig(p<{ALPHA_B:.4f})': sig
#         })
#     return pd.DataFrame(rows)

# ADDED (Reviewer 3, Comment 5): rank-biserial correlation effect size for
# the Wilcoxon signed-rank test: r = W_plus_minus_diff / total, i.e. the
# normalized difference between the sum of positive- and negative-sign
# ranks. Ranges -1 to 1; magnitude indicates effect strength.
def rank_biserial(x, y):
    diffs = np.asarray(x) - np.asarray(y)
    diffs = diffs[diffs != 0]
    if len(diffs) == 0:
        return np.nan
    ranks = pd.Series(np.abs(diffs)).rank().values
    r_plus  = ranks[diffs > 0].sum()
    r_minus = ranks[diffs < 0].sum()
    total = r_plus + r_minus
    return (r_plus - r_minus) / total if total > 0 else np.nan

# FIXED (Reviewer 3, Comment 5): reports rank-biserial effect size
# alongside W_stat and the Bonferroni-corrected p-value.
def run_wilcoxon(data, metric):
    mat = build_score_matrix(data, metric)
    if mat.empty:
        return pd.DataFrame()
    pairs = list(combinations(METHODS, 2))
    rows  = []
    for m1, m2 in pairs:
        x = mat[m1].values
        y = mat[m2].values
        n = min(len(x), len(y))
        if n < 3:
            rows.append({'Method_A': METHOD_LABEL[m1], 'Method_B': METHOD_LABEL[m2],
                         'W_stat': '-', 'p_raw': '-', 'p_bonf': '-', 'Effect_r': '-',
                         f'Sig(p<{ALPHA_B:.4f})': '-'})
            continue
        try:
            stat, p   = wilcoxon(x[:n], y[:n], zero_method='wilcox')
            p_bonf    = min(p * N_PAIRS, 1.0)
            eff_r     = rank_biserial(x[:n], y[:n])
            sig       = 'YES *' if p < ALPHA_B else 'no'
        except Exception:
            stat = p = p_bonf = eff_r = np.nan; sig = '-'
        rows.append({
            'Method_A':              METHOD_LABEL[m1],
            'Method_B':              METHOD_LABEL[m2],
            'W_stat':                round(stat, 3)   if not np.isnan(stat)   else '-',
            'p_raw':                 round(p, 6)      if not np.isnan(p)      else '-',
            'p_bonf':                round(p_bonf, 6) if not np.isnan(p_bonf) else '-',
            'Effect_r':              round(eff_r, 3)  if not np.isnan(eff_r)  else '-',
            f'Sig(p<{ALPHA_B:.4f})': sig
        })
    return pd.DataFrame(rows)

# FIXED (Reviewer 3, Comment 5): 'Significant' column renamed to
# 'Significant_Bonf' in run_friedman's output (now Bonferroni-corrected
# across omnibus tests); updated here to match.
wilcoxon_s1, wilcoxon_s2 = {}, {}

for _, row in friedman_s1.iterrows():
    if str(row['Significant_Bonf']).startswith('YES'):
        wilcoxon_s1[row['Metric']] = run_wilcoxon(s1_data, row['Metric'])

for _, row in friedman_s2.iterrows():
    if str(row['Significant_Bonf']).startswith('YES'):
        wilcoxon_s2[row['Metric']] = run_wilcoxon(s2_data, row['Metric'])

print(f"S1 significant metrics ({len(wilcoxon_s1)}): {list(wilcoxon_s1.keys())}")
print(f"S2 significant metrics ({len(wilcoxon_s2)}): {list(wilcoxon_s2.keys())}")

if wilcoxon_s1:
    m = list(wilcoxon_s1.keys())[0]
    print(f"\nExample — S1 Wilcoxon post-hoc for {m}:")
    print(wilcoxon_s1[m].to_string(index=False))

S1 significant metrics (8): ['Inertia', 'Silhouette', 'Davies_Bouldin', 'Variance_Ratio', 'Balance', 'SPD', 'DI_dist', 'Entropy']
S2 significant metrics (8): ['Inertia', 'Silhouette', 'Davies_Bouldin', 'Variance_Ratio', 'Balance', 'SPD', 'DI_dist', 'Entropy']

Example — S1 Wilcoxon post-hoc for Inertia:
Method_A Method_B  W_stat    p_raw   p_bonf  Effect_r Sig(p<0.0024)
 K-Means  Fairlet     0.0 0.000008 0.000160    -1.000         YES *
 K-Means     BFKM     1.0 0.000015 0.000320    -0.988         YES *
 K-Means      CCF     0.0 0.000008 0.000160    -1.000         YES *
 K-Means   PP-NFP    33.0 0.637870 1.000000    -0.154            no
 K-Means  PP-Gini     0.0 0.002218 0.046572    -1.000         YES *
 K-Means Rawlsian    57.0 0.228752 1.000000    -0.333            no
 Fairlet     BFKM     0.0 0.000008 0.000160     1.000         YES *
 Fairlet      CCF     0.0 0.000008 0.000160     1.000         YES *
 Fairlet   PP-NFP     0.0 0.000008 0.000160     1.000         YES *
 Fairlet  PP-Gi

In [14]:
# ADDED (Reviewer 3, Comment 5): hierarchical pairwise comparison.
# A Wilcoxon test on the eight base datasets cannot reach the Bonferroni
# threshold (minimum two-sided p = 2/2^8 = 0.0078 > 0.0024), so pairwise
# comparisons on the hierarchical matrix use the Nemenyi test after the
# Friedman test (Demsar, 2006), which controls the family-wise error rate itself.
def run_nemenyi_hier(data, metric):
    mat = build_hierarchical_matrix(data, metric)
    if mat.empty or len(mat) < 3:
        return pd.DataFrame()
    pmat = sp.posthoc_nemenyi_friedman(mat[METHODS].values)
    pmat.index = pmat.columns = METHODS
    oriented = -mat[METHODS] if metric in HIGHER else mat[METHODS]   # rank 1 = best
    mean_rank = oriented.rank(axis=1, method='average').mean()
    rows = []
    for m1, m2 in combinations(METHODS, 2):
        p = float(pmat.loc[m1, m2])
        rows.append({'Method_A': METHOD_LABEL[m1], 'Method_B': METHOD_LABEL[m2],
                     'MeanRank_A': round(mean_rank[m1], 3), 'MeanRank_B': round(mean_rank[m2], 3),
                     'p_nemenyi': round(p, 6),
                     'Effect_r': round(rank_biserial(mat[m1].values, mat[m2].values), 3),
                     'Sig(p<0.05)': 'YES *' if p < ALPHA else 'no'})
    return pd.DataFrame(rows)

with pd.ExcelWriter('Statistical_Tests_Nemenyi_Hier.xlsx') as xw:
    for tag, data in [('S1', s1_data), ('S2', s2_data)]:
        for col in ALL_COLS:
            res = run_nemenyi_hier(data, col)
            res.to_excel(xw, sheet_name=f'N_{tag}_{col}'[:31], index=False)
            n_sig = (res['Sig(p<0.05)'] == 'YES *').sum() if not res.empty else 0
            print(f"{tag} {col:15s}: {n_sig}/21 significant pairs (hierarchical Nemenyi)")
print("✓ Saved: Statistical_Tests_Nemenyi_Hier.xlsx")

S1 Inertia        : 3/21 significant pairs (hierarchical Nemenyi)
S1 Silhouette     : 2/21 significant pairs (hierarchical Nemenyi)
S1 Davies_Bouldin : 4/21 significant pairs (hierarchical Nemenyi)
S1 Variance_Ratio : 4/21 significant pairs (hierarchical Nemenyi)
S1 Balance        : 0/21 significant pairs (hierarchical Nemenyi)
S1 SPD            : 3/21 significant pairs (hierarchical Nemenyi)
S1 DI_dist        : 0/21 significant pairs (hierarchical Nemenyi)
S1 Entropy        : 2/21 significant pairs (hierarchical Nemenyi)
S2 Inertia        : 4/21 significant pairs (hierarchical Nemenyi)
S2 Silhouette     : 1/21 significant pairs (hierarchical Nemenyi)
S2 Davies_Bouldin : 5/21 significant pairs (hierarchical Nemenyi)
S2 Variance_Ratio : 4/21 significant pairs (hierarchical Nemenyi)
S2 Balance        : 0/21 significant pairs (hierarchical Nemenyi)
S2 SPD            : 2/21 significant pairs (hierarchical Nemenyi)
S2 DI_dist        : 1/21 significant pairs (hierarchical Nemenyi)
S2 Entropy

## Cell 7: Win-Loss Count

In [16]:
def win_counts(data, metrics_list, higher_set):
    """Count wins per method across all datasets × metrics."""
    wins  = {METHOD_LABEL[m]: 0 for m in METHODS}
    total = 0
    for metric in metrics_list:
        mat = build_score_matrix(data, metric)
        if mat.empty:
            continue
        for ds in mat.index:
            row_vals = {m: mat.loc[ds, m] for m in METHODS
                        if not np.isnan(mat.loc[ds, m])}
            if not row_vals:
                continue
            # FIXED: ties -- previously max()/min() gave a tied first place only to the
            # method listed first (e.g. PP-NFP always beat PP-Gini, whose fairness values
            # are identical). Every method tied for the best value now gets the win.
            best = max(row_vals.values()) if metric in higher_set else min(row_vals.values())
            for m, v in row_vals.items():
                if abs(v - best) < 1e-12:
                    wins[METHOD_LABEL[m]] += 1
            total += 1
    return wins, total

wins_s1, tot_s1 = win_counts(s1_data, ALL_COLS, HIGHER)
wins_s2, tot_s2 = win_counts(s2_data, ALL_COLS, HIGHER)

win_df = pd.DataFrame({
    'Method':  list(wins_s1.keys()),
    'S1_Wins': list(wins_s1.values()),
    'S1_%':    [f"{v/tot_s1*100:.1f}%" if tot_s1 > 0 else '-' for v in wins_s1.values()],
    'S2_Wins': [wins_s2[m] for m in wins_s1],
    'S2_%':    [f"{wins_s2[m]/tot_s2*100:.1f}%" if tot_s2 > 0 else '-' for m in wins_s1],
})
print(f"Total comparisons: S1={tot_s1}, S2={tot_s2}")
print(win_df.to_string(index=False))

Total comparisons: S1=144, S2=144
  Method  S1_Wins  S1_%  S2_Wins  S2_%
 K-Means       43 29.9%       35 24.3%
 Fairlet       58 40.3%       59 41.0%
    BFKM       15 10.4%       22 15.3%
     CCF        3  2.1%        2  1.4%
  PP-NFP       24 16.7%       24 16.7%
 PP-Gini       21 14.6%       24 16.7%
Rawlsian       18 12.5%       14  9.7%


## Cell 8: Save to Excel

In [18]:
wb   = Workbook()
wb.remove(wb.active)
thin = Side(style='thin', color='CCCCCC')
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

def df_to_ws(wb, df, sheet_name, highlight_sig=True):
    ws = wb.create_sheet(sheet_name[:31])
    hf  = PatternFill('solid', start_color='2E74B5')
    sig_fill = PatternFill('solid', start_color='FFF2CC')
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(1, ci, col)
        c.font      = Font(bold=True, color='FFFFFF', size=9, name='Arial')
        c.fill      = hf
        c.alignment = Alignment(horizontal='center', wrap_text=True)
        c.border    = bdr
        ws.column_dimensions[get_column_letter(ci)].width = max(14, len(str(col))+2)
    for ri, row in enumerate(df.itertuples(index=False), 2):
        is_sig = any('YES' in str(v) for v in row)
        for ci, val in enumerate(row, 1):
            c = ws.cell(ri, ci, val)
            c.font      = Font(size=9, name='Arial',
                               bold=('YES' in str(val)))
            c.fill      = sig_fill if (highlight_sig and is_sig) else PatternFill()
            c.alignment = Alignment(horizontal='center')
            c.border    = bdr
    ws.freeze_panes = 'A2'
    return ws

# Friedman sheets
df_to_ws(wb, friedman_s1, 'Friedman_S1')
df_to_ws(wb, friedman_s2, 'Friedman_S2')

# Win-loss
df_to_ws(wb, win_df, 'Win_Loss', highlight_sig=False)

# Wilcoxon sheets
for metric, df in wilcoxon_s1.items():
    if not df.empty:
        df_to_ws(wb, df, f'W_S1_{metric[:15]}')
for metric, df in wilcoxon_s2.items():
    if not df.empty:
        df_to_ws(wb, df, f'W_S2_{metric[:15]}')

wb.save('Statistical_Tests.xlsx')
print("✓ Saved: Statistical_Tests.xlsx")
print(f"  Sheets: Friedman_S1, Friedman_S2, Win_Loss + {len(wilcoxon_s1)} S1 Wilcoxon + {len(wilcoxon_s2)} S2 Wilcoxon")

# Also save pkl
with open('statistical_results.pkl', 'wb') as f:
    pickle.dump({
        'friedman_s1':  friedman_s1,
        'friedman_s2':  friedman_s2,
        'wilcoxon_s1':  wilcoxon_s1,
        'wilcoxon_s2':  wilcoxon_s2,
        'win_loss':     win_df,
        'alpha_bonf':   ALPHA_B,
        'n_pairs':      N_PAIRS,
    }, f)
print("✓ Saved: statistical_results.pkl")

✓ Saved: Statistical_Tests.xlsx
  Sheets: Friedman_S1, Friedman_S2, Win_Loss + 8 S1 Wilcoxon + 8 S2 Wilcoxon
✓ Saved: statistical_results.pkl
